# Sensitivity to unobserved confounding

One engine, three views. **(a)** Cinelli & Hazlett's (2020) partial-R² benchmarking prices the
bias an unobserved confounder of stated strength would add to a linear estimate — with their
published Darfur numbers reproduced below. **(b)** `shift_posterior` folds a bias distribution
into existing posterior draws without a refit. **(c)** `tipping_point` asks the decision-scale
question: how much bias flips the decision?

In [ ]:
import numpy as np

from axiom.core import Posterior, Unsupported
from axiom.diagnose import (
    Benchmark, BiasBounds, RobustnessValue, ShiftedPosterior, TippingPoint, benchmark, bias_bounds,
    partial_r2, robustness_value, shift_posterior, tipping_point,
)

# Cinelli & Hazlett (2020), Table 1: "peacefactor ~ directlyharmed + ..." in the Darfur data.
DARFUR = dict(estimate=0.0973, se=0.0232, df=783)

## Robustness values

`partial_r2` converts a t-statistic into the partial R² of the treatment with the outcome.
`robustness_value` returns `RV_q` — the strength (as partial R² with both treatment and
outcome) a confounder needs to reduce the estimate by a fraction `q` — and `RV_{q,α}`, the
strength needed to make it no longer significant at `α`. The paper reports 13.9 % and 7.6 %.

In [ ]:
print("partial R2 of t=4.19 on df=783:", round(partial_r2(0.0973 / 0.0232, 783), 4))
rv = robustness_value(**DARFUR, q=1.0, alpha=0.05)
assert isinstance(rv, RobustnessValue)
print(f"RV_1 = {rv.rv:.3f}   RV_1,0.05 = {rv.rv_alpha:.3f}   R2_yd|x = {rv.r2_yd_x:.3f}")
print("round-trips:", RobustnessValue.from_json(rv.to_json()) == rv)

## Bias bounds at a stated confounder strength

`bias_bounds` takes the confounder's partial R² with the outcome (`r2_yz_dx`) and with the
treatment (`r2_dz_x`) and returns the bias, the adjusted estimate, se, t, and a Wald interval
with its mass. The "1× female" bound from the paper (`r2_dz_x = 0.0092`, `r2_yz_dx = 0.1246`)
gives an adjusted estimate of 0.0752.

In [ ]:
b = bias_bounds(**DARFUR, r2_yz_dx=0.1246, r2_dz_x=0.0092, mass=0.95)
assert isinstance(b, BiasBounds)
print(f"bias {b.bias:.4f} -> adjusted {b.adjusted_estimate:.4f} (se {b.adjusted_se:.4f}, t {b.adjusted_t:.2f})")
print("adjusted interval:", b.adjusted_interval)
# at strength RV the confounder removes the estimate exactly
at_rv = bias_bounds(**DARFUR, r2_yz_dx=rv.rv, r2_dz_x=rv.rv)
print("adjusted estimate at RV:", round(at_rv.adjusted_estimate, 10))

## Benchmarking against an observed covariate

`benchmark` expresses a hypothetical confounder as "`k_d` times as strong as covariate `j` on
the treatment and `k_y` times on the outcome", computes the implied `(r2_dz_x, r2_yz_dx)`, and
attaches the bounds. A multiple that pushes the implied R² out of `[0, 1)` is a typed
`Unsupported`, not a wrong number.

In [ ]:
bm = benchmark(**DARFUR, covariate="female", r2_dxj_x=0.00916, r2_yxj_dx=0.11, k_d=1.0, k_y=1.0)
assert isinstance(bm, Benchmark)
print(f"1x female: r2_dz_x={bm.r2_dz_x:.4f} r2_yz_dx={bm.r2_yz_dx:.4f} adjusted={bm.bounds.adjusted_estimate:.4f}")
three = benchmark(**DARFUR, covariate="female", r2_dxj_x=0.00916, r2_yxj_dx=0.11, k_d=3.0, k_y=3.0)
assert isinstance(three, Benchmark)
print(f"3x female: adjusted={three.bounds.adjusted_estimate:.4f}")
too_strong = benchmark(**DARFUR, covariate="x", r2_dxj_x=0.4, r2_yxj_dx=0.1, k_d=3.0)
assert isinstance(too_strong, Unsupported)
print("typed failure:", too_strong.reason)

## Shifting a posterior by a bias distribution

`shift_posterior` subtracts bias draws (constant, normal with `bias_mean`/`bias_sd`, or
user-supplied) from estimate draws and returns a `ShiftedPosterior` carrying the three
summaries, the resulting interval, and a `Posterior` with `estimate`, `bias`, and `shifted`
draws.

In [ ]:
rng = np.random.default_rng(1)
draws = rng.normal(2.0, 0.5, size=4000)
s = shift_posterior(draws, bias_mean=0.5, bias_sd=0.5, mass=0.9, seed=3)
assert isinstance(s, ShiftedPosterior)
print(f"estimate {s.estimate.mean:.3f} ± {s.estimate.sd:.3f} | bias {s.bias.mean:.3f} | shifted {s.shifted.mean:.3f} ± {s.shifted.sd:.3f}")
print("interval:", s.interval, "| names:", sorted(s.names()))
post = Posterior({"tau": draws[None, :]})
s2 = shift_posterior(post, name="tau", bias_draws=[0.25, 0.5, 0.75], seed=0)
print("from a Posterior with explicit bias draws:", round(s2.shifted.mean, 3))

## Tipping points on the decision scale

`tipping_point` evaluates `P(estimate − bias > threshold)` along a bias grid (sorted by
absolute size) and reports the smallest bias at which the decision flips below `certainty`,
with the interval at that bias.

In [ ]:
tp = tipping_point(rng.normal(1.0, 0.1, size=4000), 0.5, np.linspace(-1.0, 1.0, 41), certainty=0.5)
assert isinstance(tp, TippingPoint)
print(f"decision at zero bias: {tp.decision_at_zero} (P={tp.probability_at_zero:.3f})")
print(f"flips at bias {tp.bias} with P={tp.probability:.3f}; interval there: {tp.interval}")
safe = tipping_point(np.full(100, 5.0) + np.arange(100) * 1e-3, 0.0, [0.1, 0.2])
print("no flip on the grid:", not safe.flipped, "| round-trips:", TippingPoint.from_json(tp.to_json()) == tp)